
# Collaborative Filtering





<h1>Table of contents</h1>

<div class="alert alert-block alert-info" style="margin-top: 20px">
    <ol>
        <li><a href="#ref1">Acquiring the Data</a></li>
        <li><a href="#ref2">Preprocessing</a></li>
        <li><a href="#ref3">Collaborative Filtering</a></li>
    </ol>
</div>
<br>
<hr>


<hr>

<a id="ref2"></a>

# Preprocessing


In [154]:
#Dataframe manipulation library
import pandas as pd
#Math functions, we'll only need the sqrt function so let's import only that
from math import sqrt
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [155]:
#Storing the movie information into a pandas dataframe
movies_df = pd.read_csv('movies/ml-latest/movies.csv')
#Storing the user information into a pandas dataframe
ratings_df = pd.read_csv('movies/ml-latest/ratings.csv')

In [156]:
#Head is a function that gets the first N rows of a dataframe. N's default is 5.
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [157]:
#Using regular expressions to find a year stored between parentheses
#We specify the parantheses so we don't conflict with movies that have years in their titles
movies_df['year'] = movies_df.title.str.extract('(\(\d\d\d\d\))',expand=False)
#Removing the parentheses
movies_df['year'] = movies_df.year.str.extract('(\d\d\d\d)',expand=False)
#Removing the years from the 'title' column
movies_df['title'] = movies_df.title.str.replace('(\(\d\d\d\d\))', '')
#Applying the strip function to get rid of any ending whitespace characters that may have appeared
movies_df['title'] = movies_df['title'].apply(lambda x: x.strip())

<>:3: SyntaxWarning: invalid escape sequence '\('
<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:7: SyntaxWarning: invalid escape sequence '\('
<>:3: SyntaxWarning: invalid escape sequence '\('
<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:7: SyntaxWarning: invalid escape sequence '\('
C:\Users\Arya\AppData\Local\Temp\ipykernel_21992\3811828904.py:3: SyntaxWarning: invalid escape sequence '\('
  movies_df['year'] = movies_df.title.str.extract('(\(\d\d\d\d\))',expand=False)
C:\Users\Arya\AppData\Local\Temp\ipykernel_21992\3811828904.py:5: SyntaxWarning: invalid escape sequence '\d'
  movies_df['year'] = movies_df.year.str.extract('(\d\d\d\d)',expand=False)
C:\Users\Arya\AppData\Local\Temp\ipykernel_21992\3811828904.py:7: SyntaxWarning: invalid escape sequence '\('
  movies_df['title'] = movies_df.title.str.replace('(\(\d\d\d\d\))', '')


Let's look at the result!


In [158]:
movies_df.head()

,movieId,title,genres,year
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men (1995),Comedy|Romance,1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II (1995),Comedy,1995


With that, let's also drop the genres column since we won't need it for this particular recommendation system.


In [159]:
#Dropping the genres column
movies_df = movies_df.drop('genres', axis=1)

Here's the final movies dataframe:


In [160]:
movies_df.head()

,movieId,title,year
0,1,Toy Story (1995),1995
1,2,Jumanji (1995),1995
2,3,Grumpier Old Men (1995),1995
3,4,Waiting to Exhale (1995),1995
4,5,Father of the Bride Part II (1995),1995


<br>


Next, let's look at the ratings dataframe.


In [161]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,169,2.5,1204927694
1,1,2471,3.0,1204927438
2,1,48516,5.0,1204927435
3,2,2571,3.5,1436165433
4,2,109487,4.0,1436165496


Every row in the Every row in the ratings dataframe consists of a user id associated with a minimum of one movie, a rating, and a timestamp that notes when the user rated the movie. Consequently, since we will not be utilizing the timestamp column, we will remove this column to optimize memory.ratings dataframe has a user id associated with at least one movie, a rating and a timestamp showing when they reviewed it. We won't be needing the timestamp column, so let's drop it to save on memory.


In [162]:
#Drop removes a specified row or column from a dataframe
ratings_df = ratings_df.drop('timestamp', axis=1)

Here's how the final ratings Dataframe looks like:


In [163]:
ratings_df.head()

,userId,movieId,rating
0,1,169,2.5
1,1,2471,3.0
2,1,48516,5.0
3,2,2571,3.5
4,2,109487,4.0


<hr>

<a id="ref3"></a>

# Collaborative Filtering


We are now ready to commence our discussion of recommendation systems.

Our first technique that we will discuss is called **Collaborative Filtering** (sometimes called **User-User Filtering**). As is implied by its alternate name, this technique uses other users to recommend items to the input user. In this instance, it seeks user that have similar preferences and opinions as the input user, and then recommends items that those users liked to the input user. There are a variety of techniques for calculating similar users (with some even using Machine Learning), but we will be using a technique that is based on the **Pearson Correlation Function**.


<br><br>

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%205/images/User_Item.png" width=800px>

<br><br>

The steps to produce a User Based recommendation system is the following:

*   Choose a user, and their movies that user viewed
*   Based off the user's rating to the movies find the top X most similar people
*   Find the watched movie record of the user for each of the similar people
*   Generate some similarity score by some method
*   Recommend the items with the higher score
 
Next, let's make an input user so we can recommend some movies:

Note: To add more movies, simply increase the amount of elements in the userInput list. Go ahead and add more in! Just be sure part of the movie name is capitalized, and if it has a "The" in the front of the name like, "The Matrix" then put it in like this: 'Matrix,' The' .


In [164]:
userInput = [
            {'title':'Avengers: Age of Ultron', 'rating':5},
            {'title':'Fantastic Four', 'rating':4.5},
            {'title':'Spider-Man 3', 'rating':4},
            {'title':"Amazing Spider-Man, The", 'rating':4.5},
            {'title':'Thor: The Dark World (2013)', 'rating':4.5},
            {'title':'Captain America: The First Avenger', 'rating':5},
            {'title':'Captain America: The Winter Soldier', 'rating':4.75}   
         ] 
inputMovies = pd.DataFrame(userInput)
inputMovies

,title,rating
0,Avengers: Age of Ultron,5.00
1,Fantastic Four,4.50
2,Spider-Man 3,4.00
3,"Amazing Spider-Man, The",4.50
4,Thor: The Dark World (2013),4.50
5,Captain America: The First Avenger,5.00
6,Captain America: The Winter Soldier,4.75


#### Add movieId to input user

Now that we've completed the input, let's get the ID's for the input movies from the movies dataframe and add them into the input.

This can be done by first filtering out all the rows containing the input movies' titles and then merging it to the input dataframe. We will also drop unnecessary columns for the input moive title and id to lessen memory space.


In [165]:

# Build normalized titles (remove trailing year and collapse spaces)
movies_df["norm_title"] = (
    movies_df["title"].astype(str).str.strip()
    .str.replace(r"\s\(\d{4}\)$", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)
inputMovies["norm_title"] = (
    inputMovies["title"].astype(str).str.strip()
    .str.replace(r"\s\(\d{4}\)$", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

# Filter by normalized titles and merge to get movieId with ratings
inputId = movies_df[movies_df["norm_title"].isin(inputMovies["norm_title"])]
inputMovies = pd.merge(inputId, inputMovies, on="norm_title", how="inner")

# Drop unused columns if present
inputMovies = inputMovies.drop(columns=["genres", "year"], errors="ignore")

# Final tidy frame: movieId, title (no year), rating
inputMovies = inputMovies.rename(columns={"norm_title": "title"})[["movieId", "title", "rating"]]

inputMovies

,movieId,title,rating
0,34150,Fantastic Four,4.50
1,52722,Spider-Man 3,4.00
2,88140,Captain America: The First Avenger,5.00
3,95510,"Amazing Spider-Man, The",4.50
4,106072,Thor: The Dark World,4.50
5,110102,Captain America: The Winter Soldier,4.75
6,122892,Avengers: Age of Ultron,5.00
7,122902,Fantastic Four,4.50


#### The users who has seen the same movies

Now with the movie ID's in our input, we can now get the subset of users that have watched and reviewed the movies in our input.


In [166]:
#Filtering out users that have watched movies that the input has watched and storing it
userSubset = ratings_df[ratings_df['movieId'].isin(inputMovies['movieId'].tolist())]
userSubset.head()

,userId,movieId,rating
1217,15,88140,2.5
2886,23,88140,4.5
2893,23,95510,5.0
3130,28,34150,3.5
3347,31,88140,3.0


We now group up the rows by user ID.


In [167]:
#Groupby creates several sub dataframes where they all have the same value in the column specified as the parameter
userSubsetGroup = userSubset.groupby(['userId'])

Let's look at one of the users, e.g. the one with userID=1130.


In [168]:
userSubsetGroup.get_group(1130)

C:\Users\Arya\AppData\Local\Temp\ipykernel_21992\259098224.py:1: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  userSubsetGroup.get_group(1130)


,userId,movieId,rating
105105,1130,88140,1.5


Let's also sort these groups so the users that share the most movies in common with the input have higher priority. This provides a richer recommendation since we won't go through every single user.


In [169]:
#Sorting it so users with movie most in common with the input will have priority
userSubsetGroup = sorted(userSubsetGroup,  key=lambda x: len(x[1]), reverse=True)

Now let's look at the first user.


In [170]:
userSubsetGroup[0:3]

[((2569,),
          userId  movieId  rating
  234339    2569    34150     4.0
  234415    2569    52722     4.5
  234665    2569    88140     4.5
  234751    2569    95510     4.5
  234858    2569   106072     4.0
  234886    2569   110102     2.5
  234977    2569   122892     4.5
  234979    2569   122902     1.0),
 ((3734,),
          userId  movieId  rating
  341614    3734    34150     4.0
  341767    3734    52722     3.0
  342099    3734    88140     4.0
  342153    3734    95510     3.0
  342226    3734   106072     4.0
  342252    3734   110102     4.0
  342298    3734   122892     4.0
  342300    3734   122902     3.0),
 ((7220,),
          userId  movieId  rating
  667441    7220    34150     4.0
  667511    7220    52722     5.0
  667651    7220    88140     4.0
  667673    7220    95510     3.5
  667723    7220   106072     4.0
  667749    7220   110102     4.5
  667800    7220   122892     4.5
  667802    7220   122902     1.0)]

#### Similarity of users to input user

Next, we are going to compare all users (not really all !!!) to our specified user and find the one that is most similar.\
we're going to find out how similar each user is to the input through the **Pearson Correlation Coefficient**. It is used to measure the strength of a linear association between two variables. The formula for finding this coefficient between sets X and Y with N values can be seen in the image below.

Why Pearson Correlation?

Pearson correlation is invariant to scaling, i.e. multiplying all elements by a nonzero constant or adding any constant to all elements. For example, if you have two vectors X and Y,then, pearson(X, Y) == pearson(X, 2 \* Y + 3). This is a pretty important property in recommendation systems because for example two users might rate two series of items totally different in terms of absolute rates, but they would be similar users (i.e. with similar ideas) with similar rates in various scales .

![alt text](https://wikimedia.org/api/rest_v1/media/math/render/svg/bd1ccc2979b0fd1c1aec96e386f686ae874f9ec0 "Pearson Correlation")

The values given by the formula vary from r = -1 to r = 1, where 1 forms a direct correlation between the two entities (it means a perfect positive correlation) and -1 forms a perfect negative correlation.

In our case, a 1 means that the two users have similar tastes while a -1 means the opposite.


We will select a subset of users to iterate through. This limit is imposed because we don't want to waste too much time going through every single user.


In [171]:
userSubsetGroup = userSubsetGroup[0:100]

Now, we calculate the Pearson Correlation between input user and subset group, and store it in a dictionary, where the key is the user Id and the value is the coefficient.


In [172]:
#Store the Pearson Correlation in a dictionary, where the key is the user Id and the value is the coefficient
pearsonCorrelationDict = {}

#For every user group in our subset
for name, group in userSubsetGroup:
    #Let's start by sorting the input and current user group so the values aren't mixed up later on
    group = group.sort_values(by='movieId')
    inputMovies = inputMovies.sort_values(by='movieId')
    #Get the N for the formula
    nRatings = len(group)
    #Get the review scores for the movies that they both have in common
    temp_df = inputMovies[inputMovies['movieId'].isin(group['movieId'].tolist())]
    #And then store them in a temporary buffer variable in a list format to facilitate future calculations
    tempRatingList = temp_df['rating'].tolist()
    #Let's also put the current user group reviews in a list format
    tempGroupList = group['rating'].tolist()
    #Now let's calculate the pearson correlation between two users, so called, x and y
    Sxx = sum([i**2 for i in tempRatingList]) - pow(sum(tempRatingList),2)/float(nRatings)
    Syy = sum([i**2 for i in tempGroupList]) - pow(sum(tempGroupList),2)/float(nRatings)
    Sxy = sum( i*j for i, j in zip(tempRatingList, tempGroupList)) - sum(tempRatingList)*sum(tempGroupList)/float(nRatings)
    
    #If the denominator is different than zero, then divide, else, 0 correlation.
    if Sxx != 0 and Syy != 0:
        pearsonCorrelationDict[name] = Sxy/sqrt(Sxx*Syy)
    else:
        pearsonCorrelationDict[name] = 0


In [173]:
pearsonCorrelationDict

{(2569,): 0.037488943826430775,
 (3734,): 0.662266178532522,
 (7220,): 0.005605518546712105,
 (14588,): 0.1777046633277277,
 (23383,): 0.4032441137045824,
 (27332,): -0.19389168358237033,
 (36946,): 0.051298917604257706,
 (38957,): 0.29199855803537256,
 (39699,): 0.5916244528223863,
 (40333,): 0.7170676092888448,
 (43618,): 0.46392840317216705,
 (45232,): 0.7887906006439832,
 (47648,): 0.6078656716445112,
 (51033,): 0.2416577568092735,
 (53735,): 0.7040710434833447,
 (55961,): 0.47966153480337026,
 (56914,): 0.4715595625715076,
 (59853,): 0.8668506341927302,
 (67073,): 0.40646942234853023,
 (69443,): 0.14975189244419534,
 (70987,): 0.3620388442644453,
 (73725,): 0.5893796917545018,
 (74007,): 0.2659695218051501,
 (75952,): 0.6489789085282036,
 (76703,): 0.3397130417469969,
 (78869,): 0.2539166875385041,
 (80113,): 0.6818032616569916,
 (81945,): 0.2313463350921371,
 (82597,): -0.09473684210526316,
 (83234,): 0.34500484066310094,
 (84626,): 0.8771182271731441,
 (90693,): 0.71839125107558

In [174]:
pearsonDF = pd.DataFrame.from_dict(pearsonCorrelationDict, orient='index')
pearsonDF.columns = ['similarityIndex']
pearsonDF['userId'] = pearsonDF.index
pearsonDF.index = range(len(pearsonDF))
pearsonDF.head()

,similarityIndex,userId
0,0.037489,"(2569,)"
1,0.662266,"(3734,)"
2,0.005606,"(7220,)"
3,0.177705,"(14588,)"
4,0.403244,"(23383,)"


#### The top x similar users to input user

Now let's get the top 50 users that are most similar to the input.


In [175]:
topUsers=pearsonDF.sort_values(by='similarityIndex', ascending=False)[0:50]
topUsers.head()

,similarityIndex,userId
30,0.877118,"(84626,)"
17,0.866851,"(59853,)"
11,0.788791,"(45232,)"
41,0.746219,"(117441,)"
60,0.731889,"(156725,)"


Now, let's start recommending movies to the input user.

#### Rating of selected users to all movies

We're going to do this by taking the weighted average of the ratings of the movies using the Pearson Correlation as the weight. But to do this, we first need to get the movies watched by the users in our **pearsonDF** from the ratings dataframe and then store their correlation in a new column called \_similarityIndex". This is achieved below by merging of these two tables.


In [176]:
# If userId column has tuples, extract the first element
if isinstance(topUsers['userId'].iloc[0], tuple):
    topUsers['userId'] = topUsers['userId'].apply(lambda x: x[0])

# Now force numeric type
topUsers['userId'] = pd.to_numeric(topUsers['userId'], errors='coerce')
ratings_df['userId'] = pd.to_numeric(ratings_df['userId'], errors='coerce')

# Drop NaNs
topUsers = topUsers.dropna(subset=['userId'])
ratings_df = ratings_df.dropna(subset=['userId'])

# Cast to int
topUsers['userId'] = topUsers['userId'].astype(int)
ratings_df['userId'] = ratings_df['userId'].astype(int)

# Merge safely
topUsersRating = topUsers.merge(ratings_df, on='userId', how='inner')
topUsersRating.head()


,similarityIndex,userId,movieId,rating
0,0.877118,84626,1,4.0
1,0.877118,84626,2,3.0
2,0.877118,84626,3,4.0
3,0.877118,84626,5,3.0
4,0.877118,84626,10,3.0


Now all we need to do is simply multiply the movie rating by its weight (The similarity index), then sum up the new ratings and divide it by the sum of the weights.

We can easily do this by simply multiplying two columns, then grouping up the dataframe by movieId and then dividing two columns:

It shows the idea of all similar users to candidate movies for the input user:


In [177]:
#Multiplies the similarity by the user's ratings
topUsersRating['weightedRating'] = topUsersRating['similarityIndex']*topUsersRating['rating']
topUsersRating.head()

,similarityIndex,userId,movieId,rating,weightedRating
0,0.877118,84626,1,4.0,3.508473
1,0.877118,84626,2,3.0,2.631355
2,0.877118,84626,3,4.0,3.508473
3,0.877118,84626,5,3.0,2.631355
4,0.877118,84626,10,3.0,2.631355


In [178]:
#Applies a sum to the topUsers after grouping it up by userId
tempTopUsersRating = topUsersRating.groupby('movieId').sum()[['similarityIndex','weightedRating']]
tempTopUsersRating.columns = ['sum_similarityIndex','sum_weightedRating']
tempTopUsersRating.head()

,sum_similarityIndex,sum_weightedRating
movieId,,
1,25.350303,102.163357
2,21.764596,72.860257
3,5.494372,18.293525
4,1.529117,4.587350
5,7.364200,19.696794


In [179]:
#Creates an empty dataframe
recommendation_df = pd.DataFrame()
#Now we take the weighted average
recommendation_df['weighted average recommendation score'] = tempTopUsersRating['sum_weightedRating']/tempTopUsersRating['sum_similarityIndex']
recommendation_df['movieId'] = tempTopUsersRating.index
recommendation_df.head()

,weighted average recommendation score,movieId
movieId,,
1,4.030064,1
2,3.347650,2
3,3.329503,3
4,3.000000,4
5,2.674669,5


Now let's sort it and see the top 20 movies that the algorithm recommended!


In [180]:
recommendation_df = recommendation_df.sort_values(by='weighted average recommendation score', ascending=False)
recommendation_df.head(10)

,weighted average recommendation score,movieId
movieId,,
139100,5.0,139100
6031,5.0,6031
134601,5.0,134601
6271,5.0,6271
6020,5.0,6020
5339,5.0,5339
136628,5.0,136628
136890,5.0,136890
136892,5.0,136892


In [181]:
movies_df.loc[movies_df['movieId'].isin(recommendation_df.head(10)['movieId'].tolist())]

,movieId,title,year,norm_title
5243,5339,Husbands and Wives (1992),1992,Husbands and Wives
5922,6020,Alice Adams (1935),1935,Alice Adams
5933,6031,Imitation of Life (1959),1959,Imitation of Life
6104,6202,Late Marriage (Hatuna Meuheret) (2001),2001,Late Marriage (Hatuna Meuheret)
6173,6271,Day for Night (La Nuit Américaine) (1973),1973,Day for Night (La Nuit Américaine)
29407,134601,Lost Soul: The Doomed Journey of Richard Stanl...,2014,Lost Soul: The Doomed Journey of Richard Stanl...
30071,136628,Clerks - The Flying Car (2002),2002,Clerks - The Flying Car
30171,136890,Eastern Boys (2014),2014,Eastern Boys
30172,136892,Dreams with Sharp Teeth (2008),2008,Dreams with Sharp Teeth
30705,139100,Once Brothers (2010),2010,Once Brothers


### Strengths and Weaknesses of Collaborative Filtering

##### Strengths

*   Considers the ratings of other user's
*   Does not have to analyze or gather data on the recommended item
*   Adjusts itself to the interests of the user that may change.

##### Weaknesses

*  Approximation function can take a long time
*  There may not be enough users for a meaningful approximation
*  Privacy concerns when you attempt to know user's preferences.
